[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_ConformalLSTM/VaR_CEE_ConformalLSTM.ipynb)

# VaR_CEE_ConformalLSTM

Implements LSTM neural network with conformal prediction for VaR and ES forecasting.
Uses a 60-day lookback window, 64 hidden units, retrained every 21 days on a rolling
250-day window. Conformal residuals calibrate the prediction intervals.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
ROLLING_WINDOW = 250
STRIDE = 1  # R1: full daily run
SEED = 42

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

REFIT_EVERY_LSTM = 21
W = ROLLING_WINDOW
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

## Upload Data

Upload the 10 raw CSV files for index and FX series.

In [ ]:
from google.colab import files
print("Upload the 10 raw CSV files (BET.csv, WIG20.csv, PX.csv, BUX.csv, SOFIX.csv,")
print("EURRON.csv, EURPLN.csv, EURCZK.csv, EURHUF.csv, USDBGN.csv):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files")

# Compute log returns
returns_dict = {}
for country, raw_id, series_id, stype in get_all_series():
    csv_file = f"{raw_id}.csv"
    if csv_file not in uploaded:
        print(f"  WARNING: {csv_file} not found, skipping {series_id}")
        continue
    df = pd.read_csv(csv_file, parse_dates=["Date"], index_col="Date").sort_index()
    log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
    log_ret.name = series_id
    returns_dict[series_id] = log_ret
    print(f"  {series_id}: {len(log_ret)} obs")
print(f"\nTotal: {len(returns_dict)} series")

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def train_lstm(train_data, lookback=60, epochs=30, lr=0.001):
    X, y = [], []
    for i in range(lookback, len(train_data)):
        X.append(train_data[i-lookback:i])
        y.append(train_data[i])
    if len(X) < 10:
        return None
    X = np.array(X).reshape(-1, lookback, 1)
    y = np.array(y).reshape(-1, 1)
    X_t = torch.FloatTensor(X).to(device)
    y_t = torch.FloatTensor(y).to(device)

    model = LSTMModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_t), y_t)
        loss.backward()
        optimizer.step()
    return model

In [ ]:
def run_lstm_conformal(returns, oos_dates, lookback=60):
    results = []
    model = None
    last_train_i = -REFIT_EVERY_LSTM

    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < W:
            continue
        train_vals = returns.iloc[loc - W:loc].values.astype(np.float32)

        if i - last_train_i >= REFIT_EVERY_LSTM or model is None:
            model = train_lstm(train_vals, lookback=lookback)
            last_train_i = i
            if model is None:
                continue

        model.eval()
        residuals = []
        with torch.no_grad():
            for j in range(lookback, len(train_vals)):
                x = train_vals[j-lookback:j].reshape(1, lookback, 1)
                pred = model(torch.FloatTensor(x).to(device)).item()
                residuals.append(train_vals[j] - pred)
        residuals = np.array(residuals)

        x_next = train_vals[-lookback:].reshape(1, lookback, 1)
        with torch.no_grad():
            pred_next = model(torch.FloatTensor(x_next).to(device)).item()

        y_true = returns.iloc[loc]
        row = {"date": date, "y_true": y_true}
        for alpha in VAR_ALPHAS:
            row[f"var_{alpha}"] = pred_next + np.quantile(residuals, alpha)
        q_es = np.quantile(residuals, ES_ALPHA)
        tail_r = residuals[residuals <= q_es]
        row["es_2p5pct"] = pred_next + (np.mean(tail_r) if len(tail_r) > 0 else q_es)
        results.append(row)

        if (i + 1) % 100 == 0:
            print(f"    LSTM [{i+1}/{len(oos_dates)}]")

    return pd.DataFrame(results)

In [ ]:
print("=" * 60)
print("LSTM-Conformal (rolling 250-day, refit every 21 days)")
print("=" * 60)

all_results = {}
for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        continue
    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]
    print(f"\n[{series_id}] {len(returns)} obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_lstm_conformal(returns, oos_dates)
    col_map = {f"var_{a}": f"var_{str(a).replace('0.', '')}pct" for a in VAR_ALPHAS}
    df_result = df_result.rename(columns=col_map)
    out_path = OUT_DIR / f"LSTM-Conformal_{series_id}.csv"
    df_result.to_csv(out_path, index=False, float_format="%.8f")
    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {time.time()-t0:.1f}s")

print("\n=== LSTM-Conformal complete ===")

In [ ]:
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue
    df_plot = all_results[series_id]
    var_col = "var_01pct"
    if var_col not in df_plot.columns:
        continue
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4, color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6, color="purple", linestyle="--", label="LSTM+CP VaR 1%")
    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"], df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")
    n_b = breaches.sum()
    ax.set_title(f"LSTM-Conformal VaR 1% \u2014 {series_id} (breaches: {n_b}/{len(df_plot)} = {100*n_b/len(df_plot):.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from google.colab import files
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("LSTM-Conformal_*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("LSTM_Conformal_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("LSTM_Conformal_results.zip")